# Feature Engineering: Modelo 2 (Match Predictor)

## Fase 1: Preparación de Datos Históricos
En esta primera parte cruzaremos la tabla de partidos (matches.csv) con las tablas de fuerza de equipos (teams) y árbitros (referees).
El objetivo principal es evitar el Data Leakage: no podemos usar cuántos tiros hizo un equipo en *este* partido para predecir si lo va a ganar. Debemos calcular el promedio de tiros de sus últimos N partidos.

In [ ]:
import os
# --- SOLUCIÓN DE RUTAS (CWD) 100% PORTABLE ---
# Este bloque ajusta el directorio de trabajo sin importar cómo se llame la carpeta raíz
# Garantiza que el código corra si el profesor descarga el ZIP y lo renombra.
if os.path.exists('CSV (API)') and os.path.exists('Modelo 2'):
    os.chdir('Modelo 2')
print('📁 Directorio configurado en:', os.getcwd())


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 1. Cargar dataset principal de partidos
matches = pd.read_csv('../CSV (API)/matches.csv')
# Formatear fechas correctamente e iterar cronológicamente para evitar mirar el futuro
matches['date'] = pd.to_datetime(matches['date'], format='%d/%m/%Y')
matches = matches.sort_values('date').reset_index(drop=True)

# 2. Construir tablas auxiliares 100% locales y reproducibles
# En lugar de depender del API en vivo, derivamos proxies locales desde matches.csv.
# Esto mantiene la idea de fuerza de equipo y sesgo arbitral sin romper reproducibilidad.

home_side = matches[['home_team', 'fthg', 'ftag', 'ftr']].copy()
home_side['team'] = home_side['home_team']
home_side['goals_for'] = home_side['fthg']
home_side['goals_against'] = home_side['ftag']
home_side['points'] = np.select([home_side['ftr'] == 'H', home_side['ftr'] == 'D'], [3, 1], default=0)

away_side = matches[['away_team', 'ftag', 'fthg', 'ftr']].copy()
away_side['team'] = away_side['away_team']
away_side['goals_for'] = away_side['ftag']
away_side['goals_against'] = away_side['fthg']
away_side['points'] = np.select([away_side['ftr'] == 'A', away_side['ftr'] == 'D'], [3, 1], default=0)

team_history = pd.concat([
    home_side[['team', 'goals_for', 'goals_against', 'points']],
    away_side[['team', 'goals_for', 'goals_against', 'points']]
], ignore_index=True)

teams = (
    team_history.groupby('team')
    .agg(ppg=('points', 'mean'), gf=('goals_for', 'mean'), ga=('goals_against', 'mean'))
    .reset_index()
)
teams['strength'] = 1 + 4 * (teams['ppg'] - teams['ppg'].min()) / (teams['ppg'].max() - teams['ppg'].min())
teams = teams.rename(columns={'team': 'name'})

referees = (
    matches.dropna(subset=['referee'])
    .groupby('referee')
    .agg(home_win_pct=('ftr', lambda s: 100 * (s == 'H').mean()), matches_officiated=('id', 'count'))
    .reset_index()
)

print(f"Partidos cargados listos para procesar: {matches.shape[0]}")
print(f"Equipos con rating local reproducible: {teams.shape[0]}")
print(f"Árbitros con sesgo local reproducible: {referees.shape[0]}")

# Validamos la estructura con las columnas correctas (FTHG=Full Time Home Goals)
matches[['id', 'date', 'home_team', 'away_team', 'fthg', 'ftag', 'ftr']].head()

Conclusión: Se ha logrado unificar cronológicamente los 291 partidos de la liga. Además, ya no dependemos del API en vivo: la fuerza de equipos y el sesgo arbitral se derivan localmente desde `matches.csv`, dejando el notebook mucho más reproducible.

### 1.1 Calculando Promedios Históricos Móviles
Esta es la parte más crítica del **Modelo 2**. Si predecimos un partido usando las estadísticas que ocurrieron *durante* ese partido (como los tiros que hizo), estaremos engañando al modelo (Data Leakage).
Para un predictor honesto, debemos calcular cómo venían los equipos antes del silbatazo inicial. Usaremos una ventana de los últimos 5 partidos jugados por ese equipo.

In [ ]:
def get_rolling_stats(df, team, date, window=5):
    # Filtrar partidos anteriores a la fecha seleccionada donde jugó el equipo
    past_matches = df[(df['date'] < date) & ((df['home_team'] == team) | (df['away_team'] == team))]
    past_matches = past_matches.sort_values('date').tail(window)
    
    if len(past_matches) == 0:
        return pd.Series({'goals_scored_avg': 0, 'goals_conceded_avg': 0, 'shots_target_avg': 0, 'points_avg': 0})
        
    goals_scored = 0
    goals_conceded = 0
    shots_target = 0
    points = 0
    
    for _, row in past_matches.iterrows():
        if row['home_team'] == team:
            goals_scored += row['fthg']
            goals_conceded += row['ftag']
            shots_target += row['hst']
            if row['ftr'] == 'H': points += 3
            elif row['ftr'] == 'D': points += 1
        else:
            goals_scored += row['ftag']
            goals_conceded += row['fthg']
            shots_target += row['ast']
            if row['ftr'] == 'A': points += 3
            elif row['ftr'] == 'D': points += 1
            
    n = len(past_matches)
    return pd.Series({
        'goals_scored_avg': float(goals_scored) / n,
        'goals_conceded_avg': float(goals_conceded) / n,
        'shots_target_avg': float(shots_target) / n,
        'points_avg': float(points) / n
    })

print("Función de Promedios Móviles creada.")

### 1.2 Construyendo la Matriz de Entrenamiento Definitiva
Ahora iteramos sobre todo el calendario cronológicamente para crear nuestro Dataset (X e y). Además de los promedios deportivos, extraeremos las cuotas de Bet365 (b365h, b365d, b365a) para contrastar el conocimiento de las casas de apuestas vs nuestras derivadas históricas.

In [ ]:
# 1. Crear diccionarios de mapeo rápido para la Fuerza de Equipos y el Sesgo de Árbitros
team_strength = dict(zip(teams['name'], teams['strength']))
default_team_strength = float(pd.Series(team_strength.values()).median())

referee_bias = dict(zip(referees['referee'], referees['home_win_pct']))
default_ref_bias = float(pd.to_numeric(referees['home_win_pct'], errors='coerce').dropna().median())

# Creamos listas para guardar la nueva data transformada
historical_data = []

for idx, row in matches.iterrows():
    home = row['home_team']
    away = row['away_team']
    ref = row['referee']
    
    # Extraer historial LOCAL y VISITANTE
    home_stats = get_rolling_stats(matches, home, row['date'], window=5)
    away_stats = get_rolling_stats(matches, away, row['date'], window=5)
    
    # Fuerzas estáticas y Árbitro con respaldo robusto si algún nombre no matchea
    home_str = float(team_strength.get(home, default_team_strength))
    away_str = float(team_strength.get(away, default_team_strength))
    ref_bias = float(referee_bias.get(ref, default_ref_bias))
    
    # Variables objetivo
    total_goals = row['fthg'] + row['ftag']
    match_result = row['ftr'] # 'H', 'D', 'A'
    
    historical_data.append({
        'date': row['date'],
        'home_team': home,
        'away_team': away,
        # Promedios
        'home_ppg': home_stats['points_avg'],
        'home_goals_scored': home_stats['goals_scored_avg'],
        'home_goals_conceded': home_stats['goals_conceded_avg'],
        'home_sot': home_stats['shots_target_avg'],
        'away_ppg': away_stats['points_avg'],
        'away_goals_scored': away_stats['goals_scored_avg'],
        'away_goals_conceded': away_stats['goals_conceded_avg'],
        'away_sot': away_stats['shots_target_avg'],
        # Faltantes del PDF (Equipos y Árbitros)
        'home_strength': home_str,
        'away_strength': away_str,
        'referee_home_bias': ref_bias,
        # Cuotas Bet365
        'b365h': row['b365h'],
        'b365d': row['b365d'],
        'b365a': row['b365a'],
        # Targets
        'target_total_goals': total_goals,
        'target_result': match_result
    })

df_model = pd.DataFrame(historical_data)
df_model = df_model.dropna(subset=['b365h', 'b365d', 'b365a'])

print(f"¡Matriz de entrenamiento terminada! Tamaño útil: {df_model.shape}")
df_model.tail(3)

Conclusión: Se realizo una reconstrucción de una matriz de entrenamiento en contra el Data Leakage exitosa, en cada una de nuestras filas contiene el historial puro de los 5 partidos anteriores de un equipo antes del silbatazo inicial, emparejado milimétricamente con el ojo técnico de las casas de apuestas (Cuotas Bet365).

In [ ]:
# Guardamos el dataset matemático procesado para que el cuaderno de Modelos lo consuma \n
df_model.to_csv('../Modelo 2/dataset_historico_modelo_2.csv', index=False)
print("¡Dataset guardado exitosamente como '../Modelo 2/dataset_historico_modelo_2.csv'!")